# UCS420 Cognitive Computing
## Assignment 8: NLP using Python-II

Name: Ansh Sharma  
Roll number: 1024170344

In [ ]:
import re
import string
from collections import Counter

import nltk
import matplotlib.pyplot as plt
import pandas as pd

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

## Q1. Text preprocessing and word frequency

I like learning about technology because it changes how people study and work. New apps make everyday tasks faster and easier. I enjoy reading about robots, phones, and useful computer tools. Technology can also help doctors share information and make better decisions. I think people should learn how these tools work and use them carefully. In the future, I want to explore more ideas in artificial intelligence.

In [ ]:
paragraph = (
    "I like learning about technology because it changes how people study and work. "
    "New apps make everyday tasks faster and easier. I enjoy reading about robots, phones, and useful computer tools. "
    "Technology can also help doctors share information and make better decisions. I think people should learn how these tools work and use them carefully. "
    "In the future, I want to explore more ideas in artificial intelligence."
)

lower_text = paragraph.lower()
clean_text = re.sub(r"[^a-z0-9\s]", "", lower_text)
sentence_tokens = nltk.sent_tokenize(paragraph)
word_tokens = nltk.word_tokenize(lower_text)
python_tokens = clean_text.split()

print("Lowercase text:")
print(lower_text)
print("\nText without punctuation:")
print(clean_text)
print("\nSentences:")
print(sentence_tokens)
print("\nNLTK word tokens:")
print(word_tokens)
print("\nPython split tokens:")
print(python_tokens)

`split()` separates text wherever whitespace appears, so punctuation stays attached to a word. `word_tokenize()` treats punctuation marks as separate tokens and uses language-aware tokenization rules. After cleaning punctuation, the two word lists are much more alike.

In [ ]:
stop_words = set(nltk.corpus.stopwords.words('english'))
filtered_tokens = [word for word in python_tokens if word not in stop_words]
word_frequencies = Counter(filtered_tokens)

print("Tokens after removing stopwords:")
print(filtered_tokens)
print("\nWord frequency distribution:")
print(word_frequencies.most_common())

## Q2. Stemming and lemmatization

In [ ]:
alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", paragraph.lower())
content_words = [word for word in alphabetic_words if word not in stop_words]
stemmer = nltk.stem.PorterStemmer()
lemmatizer = nltk.stem.WordNetLemmatizer()
stemmed_words = [stemmer.stem(word) for word in content_words]
lemmatized_words = [lemmatizer.lemmatize(word) for word in content_words]

comparison = pd.DataFrame({
    'Word': content_words,
    'Stemmed': stemmed_words,
    'Lemmatized': lemmatized_words
})
comparison

Porter stemming cuts words down using spelling rules, so its results can look like word fragments. Lemmatization uses a vocabulary and usually returns a recognizable base word. I would use lemmatization when readability and meaning matter, and stemming when a quick reduction in word forms is enough.

## Q3. Bag of Words and TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    "City opens a new electric bus route to reduce traffic pollution",
    "Researchers develop a battery that charges electric cars in minutes",
    "Local team wins championship after a dramatic final match"
]

count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(texts)
bow = pd.DataFrame(bow_matrix.toarray(), columns=count_vectorizer.get_feature_names_out())
bow

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()
top_keywords = {}

for row_number, text in enumerate(texts):
    scores = tfidf_matrix[row_number].toarray().ravel()
    top_indices = scores.argsort()[-3:][::-1]
    top_keywords[f'Text {row_number + 1}'] = [(tfidf_terms[i], round(scores[i], 3)) for i in top_indices]

top_keywords

The Bag of Words table records how many times each vocabulary word appears in each text. TF-IDF gives more weight to terms that help distinguish one text from the others, so the top terms summarize the electric transport stories and the sports headline.

## Q4. Jaccard and cosine similarity

In [ ]:
technology_texts = [
    "Artificial intelligence systems learn patterns from data and help people make predictions. These tools are used in health care and education. Some systems can recognize images or answer questions. They can save time, but people still need to check their results.",
    "Blockchain stores information in connected blocks and lets people share records securely. This technology is used in finance and supply chains. Each new record is linked to earlier records. It can make tracking easier, but it may require a lot of energy."
]

def preprocess_text(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

tokens_one = preprocess_text(technology_texts[0])
tokens_two = preprocess_text(technology_texts[1])
set_one = set(tokens_one)
set_two = set(tokens_two)
jaccard_similarity = len(set_one & set_two) / len(set_one | set_two)

similarity_vectorizer = TfidfVectorizer()
tfidf_pair = similarity_vectorizer.fit_transform(technology_texts)
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity_score = cosine_similarity(tfidf_pair[0:1], tfidf_pair[1:2])[0, 0]

print("First text tokens:", tokens_one)
print("Second text tokens:", tokens_two)
print("Jaccard similarity:", round(jaccard_similarity, 3))
print("Cosine similarity:", round(cosine_similarity_score, 3))

Jaccard compares the unique words shared by the two texts with all unique words present. Cosine similarity compares their TF-IDF vectors, so it also reflects how informative each shared word is. For these technology descriptions, cosine similarity gives a more useful comparison because it weights shared terms by how informative they are across the two texts, while Jaccard treats every unique word equally.

## Q5. Product review sentiment

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud

reviews = [
    "The headphones sound clear, feel comfortable, and connect to my phone quickly.",
    "The battery lasts all day and the controls are easy to use.",
    "The package arrived late and one earbud stopped working after two days.",
    "The case is small and the headphones work as expected.",
    "The price is too high for the average sound quality and weak battery."
]

review_results = []
for review in reviews:
    sentiment = TextBlob(review).sentiment
    if sentiment.polarity > 0.1:
        label = 'Positive'
    elif sentiment.polarity < -0.1:
        label = 'Negative'
    else:
        label = 'Neutral'
    review_results.append({
        'Review': review,
        'Polarity': round(sentiment.polarity, 3),
        'Subjectivity': round(sentiment.subjectivity, 3),
        'Class': label
    })

review_table = pd.DataFrame(review_results)
review_table

In [ ]:
positive_reviews = ' '.join(review_table.loc[review_table['Class'] == 'Positive', 'Review'])
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(positive_reviews)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

Polarity ranges from negative to positive, while subjectivity indicates how opinion-based a review is. The labels use a small neutral range around zero. The word cloud shows the most prominent words from reviews classified as positive.

## Q6. Next-word text generation with an LSTM

In [ ]:
training_text = (
    "Learning to code takes patience and regular practice. Each small program teaches a new idea. "
    "When a student writes code, mistakes are part of the process. Reading an error message can show where a problem began. "
    "After fixing the problem, the program becomes easier to understand. Simple projects help students connect different ideas. "
    "They can build a calculator, organize information, or make a small game. Working with classmates also brings new ways to solve a task. "
    "A clear plan saves time, but trying a new approach can lead to better results. With practice, students become more confident and creative. "
    "They learn to explain their choices and improve their work step by step. Coding is useful because it turns ideas into tools that people can use."
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import numpy as np

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
sequences = tokenizer.texts_to_sequences([training_text])[0]

input_sequences = []
for index in range(1, len(sequences)):
    input_sequences.append(sequences[:index + 1])

max_sequence_length = max(len(sequence) for sequence in input_sequences)
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_length, padding='pre')
x_train = padded_sequences[:, :-1]
y_train = to_categorical(padded_sequences[:, -1], num_classes=len(tokenizer.word_index) + 1)

print("Number of words in training text:", len(sequences))
print("Vocabulary size:", len(tokenizer.word_index))
print("Number of training sequences:", len(input_sequences))

In [ ]:
model = Sequential([
    Embedding(len(tokenizer.word_index) + 1, 32, input_length=max_sequence_length - 1),
    LSTM(48),
    Dense(len(tokenizer.word_index) + 1, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=80, verbose=0)

In [ ]:
def generate_text(seed_text, next_words=10):
    result = seed_text
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([result])[0]
        sequence = pad_sequences([sequence], maxlen=max_sequence_length - 1, padding='pre')
        probabilities = model.predict(sequence, verbose=0)[0]
        next_index = int(np.argmax(probabilities))
        next_word = tokenizer.index_word.get(next_index, '')
        if not next_word:
            break
        result += ' ' + next_word
    return result

seeds = ['learning', 'students', 'coding']
for seed in seeds:
    print(generate_text(seed, 10).capitalize() + '.')

The model learns next-word patterns from a short paragraph, so the generated lines may repeat phrases or sound less natural than the training text. A larger and more varied training set would usually produce better results.